<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 11 — Search and Hashing Reference

A lookup guide. Notebooks 01 and 02 are meant to be read in order; this one is
meant to be **searched**. Keep it open while you work on Homework 10.

Run the setup cell, then jump to the section you need.

| Section | When you need it |
|---|---|
| [A — What each operation costs](#A) | choosing between a list, a sorted list, a dict and a set |
| [B — Searching a list](#B) | `in`, `index`, `count`, and their cost |
| [C — Writing a linear search](#C) | with a counter |
| [D — Writing a binary search](#D) | the loop, the promise, the two classic bugs |
| [E — Where would it go: `leftmost` and `bisect`](#E) | insertion points, ranges, prefixes |
| [F — Sorting](#F) | `sorted`, `key=`, what it costs |
| [G — Hash functions](#G) | writing one; Python's `hash`; what can be a key |
| [H — A hash table by hand](#H) | buckets, put, get, load factor, resize |
| [I — `dict` and `set`](#I) | the built-in hash tables |
| [J — Counting and grouping](#J) | tallies, `Counter`, `defaultdict` |
| [K — The same ideas in pandas](#K) | `isin`, `merge`, `searchsorted` |
| [L — Symptoms and causes](#L) | endless loops, wrong answers, slow programs |

**In Homework 10, Phases 1 to 6, you build these tools yourself.** The
built-ins of Sections B, E and I are here so that you recognise what you
built, and for use everywhere else.

In [ ]:
import bisect
import math
from collections import Counter, defaultdict

cities = ["Tampa", "Reno", "Boise", "Tucson", "Nashville", "Philadelphia",
          "Indianapolis", "Edmonton", "St. Louis", "New Orleans"]
in_order = sorted(cities)
print(in_order)

---

<a id="A"></a>
## A — What each operation costs

How the work grows with the number of items n. Constants are left out: "n"
means "proportional to n".

| Operation | Unsorted list | Sorted list | `dict` / `set` |
|---|---|---|---|
| is `x` here? | n | log n | 1 |
| value stored under key `x` | n | log n | 1 |
| add an item | 1 (`append`) | n (`insort`: items must move) | 1 |
| remove an item | n | n | 1 |
| smallest / largest | n | 1 | n |
| the next larger than `x` | n | log n | n |
| how many between `a` and `b` | n | log n | n |
| all items in order | n log n (sort) | n | n log n (sort) |
| build from n items | n | n log n | n |

| Name | Grows like | 1,000 items | 1,000,000 items |
|---|---|---:|---:|
| constant | 1 | 1 | 1 |
| logarithmic | log2 n | 10 | 20 |
| linear | n | 1,000 | 1,000,000 |
| "n log n" | n × log2 n | 10,000 | 20,000,000 |
| quadratic | n × n | 1,000,000 | 1,000,000,000,000 |

Quadratic is what a linear search inside a loop over the same data costs. It
is the usual reason a program that worked on a sample never finishes on the
full file.

---

<a id="B"></a>
## B — Searching a list

All of these are linear searches: they look at items one by one.

| You want | Write this | Cost |
|---|---|---|
| is it there? | `x in items` | n |
| its first position | `items.index(x)` (a `ValueError` when absent) | n |
| how often it occurs | `items.count(x)` | n |
| the first item that satisfies a condition | `next((item for item in items if condition), None)` | n |
| is any / are all items ... | `any(...)`, `all(...)` | n; stop early |
| smallest, largest | `min(items)`, `max(items)` | n |
| position of the smallest | `items.index(min(items))` | n |

In [ ]:
print("Boise" in cities, cities.index("Boise"), cities.count("Boise"))
print(next((city for city in cities if city.startswith("N")), None))
print(next((city for city in cities if city.startswith("X")), None))

Fine once. Inside a loop over another large collection, turn the list into a
`set` (membership) or a `dict` (lookup) first:

```python
known = set(customer_ids)               # once
for order in orders:
    if order["customer"] in known:      # one step each
        ...
```

---

<a id="C"></a>
## C — Writing a linear search

```python
def linear_search(items, target):
    comparisons = 0
    for index in range(len(items)):
        comparisons += 1
        if items[index] == target:
            return (index, comparisons)
    return (-1, comparisons)
```

| Case | Comparisons |
|---|---|
| target is first | 1 |
| target at position `i` | `i + 1` |
| target is there, on average | about n / 2 |
| target is not there | n |

`for index, item in enumerate(items):` is the same loop with the item at hand.
Return from inside the loop as soon as it is found; the code after the loop is
the "not found" case.

---

<a id="D"></a>
## D — Writing a binary search

**The list must be sorted in ascending order.** On unsorted data the result is
wrong and nothing warns you.

In [ ]:
def binary_search(items, target):
    low, high = 0, len(items) - 1          # the target, if present, is in items[low..high]
    comparisons = 0
    while low <= high:                     # something is left
        middle = (low + high) // 2
        comparisons += 1
        if items[middle] == target:
            return (middle, comparisons)
        if items[middle] < target:
            low = middle + 1               # past middle: it was looked at
        else:
            high = middle - 1              # past middle, the other way
    return (-1, comparisons)               # low has passed high: nothing left


print(binary_search(in_order, "Reno"), binary_search(in_order, "Atlanta"))

Reading it:

| Line | Why |
|---|---|
| `low, high = 0, len(items) - 1` | both ends are positions that exist; an empty list gives `high = -1` and the loop does not run |
| `while low <= high` | `<=`, because when they are equal one item is left to look at |
| `middle = (low + high) // 2` | whole-number division |
| `low = middle + 1`, `high = middle - 1` | `middle` has been ruled out, so step past it |
| `return (-1, ...)` after the loop | not found |

**The promise** (the loop invariant): if the target is in the list, it is
between `low` and `high`. Every round keeps it and shrinks the range. When you
are unsure about a `+ 1`, ask whether the line keeps the promise.

| Bug | What happens | Fix |
|---|---|---|
| `low = middle` | endless loop once `low` and `high` are neighbours | `middle + 1` |
| `high = middle` with `while low <= high` | endless loop when `low == high` | `middle - 1` |
| `while low < high` with `high = len(items) - 1` | the last remaining item is never looked at | `<=` |
| `high = len(items)` with this loop | `items[middle]` can be out of range | `len(items) - 1` |
| slicing: `items = items[middle + 1:]` | copies half the list each round, which is linear work, and the index is lost | move `low` and `high` |
| searching an unsorted list | wrong answers, no error | sort first, or do not use it |

Test with: an empty list, one item, two items, the first item, the last, a
target below everything, above everything, and between two items.

Rounds needed for n items: at most `int(math.log2(n)) + 1`.

In [ ]:
for size in (1, 2, 10, 1_000, 1_000_000, 1_000_000_000):
    print("{:>13,} items: at most {:>2} rounds".format(size, int(math.log2(size)) + 1))

---

<a id="E"></a>
## E — Where would it go: `leftmost` and `bisect`

`leftmost(items, target)` is the first position whose item is **not below**
the target. Python's name for it is `bisect.bisect_left`.

In [ ]:
def leftmost(items, target):
    low, high = 0, len(items)              # the answer is in low..high, both included
    comparisons = 0
    while low < high:
        middle = (low + high) // 2
        comparisons += 1
        if items[middle] < target:
            low = middle + 1               # middle and everything before it: too small
        else:
            high = middle                  # middle may be the answer: keep it
    return (low, comparisons)


scores = [10, 20, 20, 20, 30, 40]
for target in (5, 20, 25, 40, 99):
    print("{:2}: leftmost {}   bisect_left {}   bisect_right {}".format(
        target, leftmost(scores, target)[0], bisect.bisect_left(scores, target),
        bisect.bisect_right(scores, target)))

Differences from `binary_search`: `high` starts at `len(items)` (one past the
end is a possible answer), the loop runs `while low < high`, there is no early
return, and the upper end moves to `middle`, not `middle - 1`.

| You want (sorted `items`) | Write this |
|---|---|
| first position not below `x` | `bisect.bisect_left(items, x)` |
| first position above `x` | `bisect.bisect_right(items, x)` |
| is `x` there? | `i = bisect_left(items, x)`; then `i < len(items) and items[i] == x` |
| how often `x` occurs | `bisect_right(items, x) - bisect_left(items, x)` |
| how many are below `x` | `bisect_left(items, x)` |
| how many with `a <= item < b` | `bisect_left(items, b) - bisect_left(items, a)` |
| how many with `a <= item <= b` | `bisect_right(items, b) - bisect_left(items, a)` |
| those items | `items[bisect_left(items, a):bisect_left(items, b)]` |
| the largest item not above `x` | `items[bisect_right(items, x) - 1]` (check the index is not −1) |
| the smallest item not below `x` | `items[bisect_left(items, x)]` (check it is not `len(items)`) |
| strings starting with a prefix `p` | from `bisect_left(items, p)` to `bisect_left(items, p + "\uffff")` |
| insert and keep sorted | `bisect.insort(items, x)` (finding is log n; making room is n) |
| search by a field | `bisect_left(records, x, key=lambda r: r["age"])` (Python 3.10+) |

In [ ]:
names = ["Ana", "Bea", "Ben", "Bo", "Chi", "Dev"]

first = bisect.bisect_left(names, "B")
past = bisect.bisect_left(names, "B" + "\uffff")
print("start with B:", names[first:past])

print("largest score not above 35:", scores[bisect.bisect_right(scores, 35) - 1])
print("how many 20s:", bisect.bisect_right(scores, 20) - bisect.bisect_left(scores, 20))

`bisect` never tells you whether the value is there. It tells you where it
would go. The check is yours.

---

<a id="F"></a>
## F — Sorting

| You want | Write this |
|---|---|
| a new sorted list | `sorted(items)` |
| sort the list itself | `items.sort()` (returns `None`) |
| largest first | `sorted(items, reverse=True)` |
| by a property | `sorted(records, key=lambda r: r["stars"])` |
| by two properties | `key=lambda r: (r["city"], r["name"])` |
| one ascending, one descending (numbers) | `key=lambda r: (r["city"], -r["stars"])` |
| ignoring case | `key=str.lower` |
| the positions in sorted order | `sorted(range(len(items)), key=lambda i: items[i])` |
| the five smallest | `sorted(items)[:5]`, or `heapq.nsmallest(5, items)` |
| is it sorted? | `all(items[i] <= items[i + 1] for i in range(len(items) - 1))` |

In [ ]:
order = sorted(range(len(cities)), key=lambda position: cities[position])
print(order)
print([cities[position] for position in order])

- Sorting n items costs about n × log2(n) comparisons: 10,000 for a thousand
  items, 20 million for a million.
- Python's sort is **stable**: items that compare equal keep their order.
  Sort by the second key first, then by the first.
- Text sorts by character code: all capitals come before all lower-case
  letters, and `"10" < "9"`. Dates written `YYYY-MM-DD` sort correctly as
  text.
- When does sorting pay? When the lookups it speeds up save more than it cost:

```text
sort_cost + k * binary_each  <  k * linear_each
k  >  sort_cost / (linear_each - binary_each)
```

---

<a id="G"></a>
## G — Hash functions

A hash function turns a key into a whole number. A good one is

- **deterministic**: the same key always gives the same number;
- **spread out**: different keys rarely give the same number;
- **quick** to compute.

In [ ]:
def hash_key(key, base=31, modulus=1_000_000_007):
    value = 0
    for character in key:
        value = (value * base + ord(character)) % modulus
    return value


def sum_of_codes(key):
    return sum(ord(character) for character in key)


for word in ("listen", "silent", "ab", "ba"):
    print("{:7} polynomial {:>11,}   sum of codes {}".format(
        word, hash_key(word), sum_of_codes(word)))

| Piece | Does |
|---|---|
| `ord(character)` | the character's code: `ord("a")` is 97; `chr(97)` is `"a"` |
| `value * base + ...` | shifts what came before, so the order of characters matters |
| `% modulus` | keeps the number bounded |
| `hash_key(key) % buckets` | folds it onto the bucket numbers 0 to `buckets - 1` |

Bad hash functions, and why:

| Function | Problem |
|---|---|
| `len(key)` | all keys of one length collide |
| `ord(key[0])` | all keys with one first letter collide |
| sum of the codes | anagrams collide; sums are small numbers |
| a random number | not deterministic: nothing is ever found again |

### Python's `hash`

| Fact | Consequence |
|---|---|
| `hash("text")` differs between runs of Python | never store it or compare it across runs |
| `hash(3)` is 3; equal numbers hash equal: `hash(1) == hash(1.0) == hash(True)` | `1`, `1.0` and `True` are the same dict key |
| only **hashable** objects have one | they can be keys of a `dict` and members of a `set` |

| Hashable (can be a key) | Not hashable |
|---|---|
| `int`, `float`, `str`, `bool`, `None` | `list` |
| a `tuple` of hashable things | `dict` |
| `frozenset` | `set` |

To use a list as a key, convert it: `tuple(the_list)`. For a set,
`frozenset(the_set)`.

In [ ]:
print(hash(1) == hash(1.0) == hash(True), {1: "a", 1.0: "b", True: "c"})

pairs = {}
pairs[("Tampa", "FL")] = 1
try:
    pairs[["Tampa", "FL"]] = 1
except TypeError as error:
    print("TypeError:", error)

---

<a id="H"></a>
## H — A hash table by hand

A table is a list of **buckets**; a bucket is a list of `[key, value]` pairs.

In [ ]:
def new_table(buckets=8):
    return [[] for number in range(buckets)]            # NOT [[]] * buckets


def table_put(table, key, value):
    bucket = table[hash_key(key) % len(table)]
    for pair in bucket:
        if pair[0] == key:
            pair[1] = value                             # replace
            return False
    bucket.append([key, value])                         # new
    return True


def table_get(table, key):
    bucket = table[hash_key(key) % len(table)]
    comparisons = 0
    for pair in bucket:
        comparisons += 1
        if pair[0] == key:
            return (pair[1], comparisons)
    return (None, comparisons)


def table_resize(table):
    pairs = [pair for bucket in table for pair in bucket]
    table[:] = new_table(2 * len(table))                # same list, new contents
    for key, value in pairs:
        table_put(table, key, value)


table = new_table()
for position, city in enumerate(cities):
    table_put(table, city, position)
for number, bucket in enumerate(table):
    print(number, bucket)
print(table_get(table, "Boise"), table_get(table, "Atlanta"))

| Term | Means |
|---|---|
| bucket | one slot of the table; holds the pairs whose keys hash there |
| collision | two different keys in the same bucket |
| chaining | handling collisions by keeping a list in each bucket |
| load factor | pairs ÷ buckets: the average length of a bucket |
| resize (rehash) | more buckets, and every pair placed again |

| Pitfall | What happens | Fix |
|---|---|---|
| `[[]] * 8` | one list under eight names: every bucket shows every pair | `[[] for _ in range(8)]` |
| appending without looking first | the same key stored twice; `get` returns the old value | search the bucket, replace if found |
| `table = new_table(...)` inside `table_resize` | the caller's table is unchanged | `table[:] = ...` |
| growing without re-placing the pairs | keys sit in buckets where no lookup looks | put every pair again |
| counting all pairs after every put | building n keys costs n × n | keep a running count |
| using `hash()` for strings | another layout every run | your own hash function |

Why doubling? Each resize moves every pair, but the next one is twice as far
away. Over n puts the moves add up to less than 2n: about two extra steps per
key, on average, however large the table gets.

In [ ]:
table = new_table()
stored = 0
for number in range(1000):
    if table_put(table, "key-{}".format(number), number):
        stored += 1
    if stored / len(table) > 0.75:
        table_resize(table)

lengths = [len(bucket) for bucket in table]
print("{} buckets, load factor {:.2f}, longest bucket {}".format(
    len(table), stored / len(table), max(lengths)))

---

<a id="I"></a>
## I — `dict` and `set`

Python's hash tables. Everything in the left column is about one step,
whatever the size.

| You want | `dict` |
|---|---|
| store | `d[key] = value` |
| look up (a `KeyError` when absent) | `d[key]` |
| look up with a fallback | `d.get(key)`, `d.get(key, default)` |
| is the key there? | `key in d` |
| remove | `del d[key]`, `d.pop(key, None)` |
| look up, creating it when absent | `d.setdefault(key, [])` |
| how many | `len(d)` |
| go through it | `for key, value in d.items():` |
| build from pairs | `dict(pairs)`, `{k: v for k, v in pairs}` |
| build a lookup from records | `{record["id"]: record for record in records}` |

| You want | `set` |
|---|---|
| make one | `set(items)`, `{1, 2, 3}`; the empty one is `set()`, not `{}` |
| add, remove | `s.add(x)`, `s.discard(x)` |
| is it there? | `x in s` |
| in both | `a & b` |
| in either | `a \| b` |
| in `a` and not in `b` | `a - b` |
| in exactly one | `a ^ b` |
| is every member of `a` in `b`? | `a <= b` |
| how many different values | `len(set(items))` |
| the values without duplicates, order kept | `list(dict.fromkeys(items))` |

In [ ]:
have = {"Tampa", "Reno", "Boise"}
want = {"Reno", "Boise", "Tucson", "Atlanta"}

print("in both:        ", have & want)
print("missing:        ", want - have)
print("not wanted:     ", have - want)
print("without repeats:", list(dict.fromkeys(["b", "a", "b", "c", "a"])))

- A `dict` keeps its keys in the order they were first inserted. A `set` has
  no order you may rely on. Neither is sorted: use `sorted(d)` for that.
- `x in d` looks among the **keys**. `x in d.values()` is a linear search.
- Looking up by value needs a second dictionary, the other way round.
- A `dict` uses several times the memory of a list of the same items. That is
  the price of the speed.

---

<a id="J"></a>
## J — Counting and grouping

A tally per key is a lookup per item, so it lives in a dictionary.

| You want | Write this |
|---|---|
| how often each value occurs | `Counter(items)` |
| the three most common | `Counter(items).most_common(3)` |
| the same by hand | `tally[key] = tally.get(key, 0) + 1` |
| a sum per key | `total[key] = total.get(key, 0) + amount` |
| the items of each group | `groups.setdefault(key, []).append(item)` |
| the same with a default | `groups = defaultdict(list)`; `groups[key].append(item)` |
| the distinct values per key | `defaultdict(set)`; `seen[key].add(value)` |
| have I seen this before? | `if x in seen: ...` / `seen.add(x)` with `seen = set()` |
| the duplicates | `[x for x, n in Counter(items).items() if n > 1]` |

In [ ]:
visits = [("Tampa", 3), ("Reno", 1), ("Tampa", 2), ("Boise", 5), ("Reno", 4)]

total = {}
groups = defaultdict(list)
for city, nights in visits:
    total[city] = total.get(city, 0) + nights
    groups[city].append(nights)

print(total)
print(dict(groups))
print(Counter(city for city, nights in visits).most_common(2))

---

<a id="K"></a>
## K — The same ideas in pandas

| Plain Python | pandas | Inside |
|---|---|---|
| `x in a_set`, for a whole column | `column.isin(values)` | a hash table |
| a dictionary lookup per row | `left.merge(right, on="key")` | a hash table of one side's keys |
| a dictionary lookup per row | `column.map(a_dict)` | a hash lookup per value |
| a tally per key | `column.value_counts()`, `groupby(...).size()` | a hash table |
| `len(set(items))` | `column.nunique()` | a hash table |
| remove repeats | `column.drop_duplicates()` | a hash table |
| `bisect_left(sorted_items, x)` | `sorted_column.searchsorted(x)` | binary search |
| a row by its label | `table.set_index("id").loc[key]` | a hash table, built once |
| the rows between two labels | `table.sort_index().loc[a:b]` | binary search on a sorted index |

Slow in pandas for the same reason as in plain Python:

```python
for _, row in orders.iterrows():                         # a loop over rows ...
    match = customers[customers["id"] == row["customer"]]    # ... with a scan inside
```

`orders.merge(customers, ...)` does the same in one pass.

---

<a id="L"></a>
## L — Symptoms and causes

| Symptom | Likely cause | Fix |
|---|---|---|
| a binary search never returns | `low = middle` or `high = middle` in a `while low <= high` loop | step past `middle` |
| it misses the last item, or a list of one | `while low < high` with `high = len(items) - 1` | `<=` |
| `IndexError` in a binary search | `high = len(items)` in the `<=` version | `len(items) - 1` |
| "not found" for something that is there | the list is not sorted, or sorted by another key (case, text against number) | sort by exactly what you compare |
| `bisect_left` gives a position, and the item there is not the target | `bisect` does not test for presence | check `items[i] == x` |
| `leftmost` is slow on many equal items | walking left from a match | the `high = middle` loop |
| every bucket shows every key | `[[]] * n` | a list comprehension |
| a key appears twice in the table | `put` appends without searching | replace when found |
| keys vanish after resizing | pairs not placed again, or a new table that the caller never sees | re-put; `table[:] = ...` |
| the table is correct and lookups are slow | load factor far above 1, or a hash function that clumps | resize; a better hash |
| a different table layout on every run | Python's `hash()` on strings | your own hash function |
| `TypeError: unhashable type: 'list'` | a list as a key or set member | `tuple(...)` |
| `KeyError` | `d[key]` for a key that is absent | `d.get(key)`, or `key in d` first |
| `{}` behaves oddly as a set | `{}` is an empty **dict** | `set()` |
| a program that was fine on 1,000 rows never finishes | a linear search inside a loop | a `set` or `dict` built before the loop |
| `"10"` sorts before `"9"` | numbers stored as text | convert, or `key=int` |